## Setup and imports

In [ ]:
EXP_NAME = "11MAY_simple_synth_biased_classbal_2"
dataset = "synth/9MAY/simple_biased_classbal_train"
feature_map = "sps/features_synth_simple_biased"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'

In [ ]:
print(PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.feature_selection import mutual_info_regression
from sklearn.utils import resample

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit_baseline.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/data/{dataset}.csv')

with open(f"{PROJECT_ROOT}/configs/{feature_map}.json", 'r') as f:
  features = json.load(f)

In [ ]:
# sps_audit.head()

# Coverage

In [ ]:
iteration_per_feature = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('feature')[['iteration']].count()
iteration_per_feature

In [ ]:
full_audit = pd.concat([sps_audit, sbs_audit_baseline], ignore_index=True, axis=0)

In [ ]:
# CHANCE LEVEL AUPRC FOR REFERENCE
positives = dataset[dataset[features["target"]["name"]] == 1]
auprc_chance = len(positives) / len(dataset)

In [ ]:
# Group MACE and Max Group MACE
group_mace_cols = full_audit.columns.str.startswith('mace_')
full_audit['max_mace'] = full_audit.loc[:, group_mace_cols].max(axis=1)

# Downstream AUPRC Delta
# i.e. the loss (if negative) or gain (if positive) in utility in Udesc compared to Xdesc for a downstream classifier
full_audit['delta_ext_auprc'] = (full_audit['ext_u_auprc'] - full_audit['ext_x_auprc']) / (full_audit['ext_x_auprc'] - auprc_chance)

# Config-level trade-off analyses

In [ ]:
tradeoff_audit = full_audit.groupby('iteration')[['te_error', 'max_mace', 'internal_auprc', 'delta_ext_auprc']].median().round(3)
x_desc_configs = full_audit[full_audit['bucket'] == 'x_desc'].groupby(['iteration'])['feature'].apply(set).to_dict()

In [ ]:
agg_sbs_audit_baseline = sbs_audit_baseline.groupby('iteration')

baseline_te_error = tradeoff_audit.loc[-1, "te_error"]
baseline_max_mace = tradeoff_audit.loc[-1, "max_mace"]
baseline_internal_auprc = tradeoff_audit.loc[-1, "internal_auprc"]
baseline_delta_ext_auprc = tradeoff_audit.loc[-1, "delta_ext_auprc"]

In [ ]:
print(f"Baseline TE error: {baseline_te_error:.3f}")
print(f"Baseline Internal Average Precision: {baseline_internal_auprc:.3f}")
print(f"Baseline Downstream Average Precision Cost/Gain: {baseline_delta_ext_auprc:.3f}")
print(f"Baseline max MACE: {baseline_max_mace:.3f}")

## Utility vs Fairness trade-off

In [ ]:
utility_vs_fairness = tradeoff_audit.sort_values('internal_auprc', ascending=False)

print(f'Chance-level AUPRC: {auprc_chance:.3f}')

pareto_frontier = []
current_min_mace = 1

print('\n--- Configurations on the Internal AUPRC / max group MACE Pareto Frontier ---')
for idx, solution in utility_vs_fairness.iterrows():
  if solution['max_mace'] < current_min_mace:
    solution['delta_mace'] = (solution['max_mace'] - baseline_max_mace) / baseline_max_mace
    solution['delta_internal_auprc'] = (solution['internal_auprc'] - baseline_internal_auprc) / (baseline_internal_auprc - auprc_chance)
    solution['FEU'] = (solution['delta_internal_auprc'] / solution['delta_mace']).round(3) if solution['delta_mace'] else "NA"
    pareto_frontier.append(solution)
    current_min_mace = solution['max_mace']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs.get(idx, "empty")}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df.drop('delta_ext_auprc', axis=1).to_markdown())


fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(baseline_internal_auprc, baseline_max_mace, marker="D", color="#D92B89", linestyle="")
sns.lineplot(data=pareto_frontier_df, x='internal_auprc', y='max_mace', color="green", marker='', linestyle="--", errorbar=None, ax=ax)
sns.scatterplot(data=utility_vs_fairness, x='internal_auprc', y='max_mace', ax=ax)
ax.axvline(auprc_chance)
plt.xlabel('Internal Average Precision (AUPRC)')
plt.ylabel('Max group MACE')
plt.ylim(bottom=0)
plt.legend(labels=[ 
                   'Bias-unaware configuration',
                   'Pareto frontier', 
                   'CEVAE-HE result for each pathway configuration in the audit',
                   'Chance-level AUPRC'
                   ],
           loc='upper left', bbox_to_anchor=(0, -0.1), edgecolor="white")

plt.show()

## Downstream AUPRC Delta vs Internal Fairness trade-off

In [ ]:
utility_vs_fairness = tradeoff_audit.sort_values('delta_ext_auprc', ascending=False)

pareto_frontier = []
current_min_mace = 1

print('\n--- Configurations on the Downstream Average Precision Cost/Gain / max group MACE Pareto Frontier ---')
for idx, solution in utility_vs_fairness.iterrows():
  if solution['max_mace'] < current_min_mace:
    solution['delta_mace'] = (solution['max_mace'] - baseline_max_mace) / baseline_max_mace
    solution['FEU'] = (solution['delta_ext_auprc'] / solution['delta_mace']).round(3) if solution['delta_mace'] else "NA"
    pareto_frontier.append(solution)
    current_min_mace = solution['max_mace']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs.get(idx, "empty")}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df.to_markdown())


fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(baseline_delta_ext_auprc, baseline_max_mace, marker="D", color="#D92B89", linestyle="")
sns.lineplot(data=pareto_frontier_df, x='delta_ext_auprc', y='max_mace', color="green", marker='', linestyle="--", errorbar=None, ax=ax)
sns.scatterplot(data=utility_vs_fairness, x='delta_ext_auprc', y='max_mace', ax=ax)
plt.xlabel('Downstream Average Precision Cost/Gain')
plt.ylabel('Max group MACE')
plt.ylim(bottom=0)
plt.legend(labels=[ 
                   'Bias-unaware configuration',
                   'Pareto frontier', 
                   'CEVAE-HE result for each pathway configuration in the audit'
                   ],
           loc='upper left', bbox_to_anchor=(0, -0.1), edgecolor="white")

plt.show()

# All Configs explored

In [ ]:
def find_config(row):
  return x_desc_configs.get(row['id'], "empty Xdesc")

all_configs = utility_vs_fairness.sort_values('max_mace').reset_index(names="id")
all_configs['Xdesc config'] = all_configs.apply(find_config, axis=1)

lowest_te_error = all_configs.sort_values('te_error').reset_index().loc[0, :]
lowest_max_mace = all_configs.sort_values('max_mace').reset_index().loc[0, :]
highest_auprc  = all_configs.sort_values('internal_auprc', ascending=False).reset_index().loc[0, :]
highest_auprc_delta  = all_configs.sort_values('delta_ext_auprc', ascending=False).reset_index().loc[0, :]
if highest_auprc_delta['id'] == '-1':
  highest_auprc_delta  = all_configs.sort_values('delta_ext_auprc', ascending=False).reset_index().loc[1, :]

print(f'Config with the lowest TE error: {lowest_te_error['Xdesc config']}\n')
print(f'Config with the lowest max MACE: {lowest_max_mace['Xdesc config']}\n')
print(f'Config with the highest Internal AUPRC: {highest_auprc['Xdesc config']}\n')
print(f'Config with the lowest Downstream AUPRC Cost: {highest_auprc_delta['Xdesc config']}\n')
print(all_configs.to_markdown())